# Traductor de EPUB (Google Colab)

Traduce un libro EPUB párrafo por párrafo en una GPU de Colab, con [TranslateGemma](https://huggingface.co/google/translategemma-4b-it) (4B o 12B) o [Gemma 4 26B-A4B](https://huggingface.co/google/gemma-4-26B-A4B-it), que da la mejor calidad, y arma un EPUB **completo o parcial** con lo traducido.

- **Conserva la estructura:** cada párrafo se traduce entero; cursiva y negrita viajan como Markdown (`*x*`, `**x**`) y se reconstruyen; links y notas al pie se conservan.
- **Retomable:** cada traducción se guarda al instante en Google Drive. Si Colab se desconecta, vuelve a ejecutar las celdas y sigue donde quedó.
- **EPUB parcial:** la celda 7 arma un EPUB con los capítulos ya traducidos, aunque la traducción no haya terminado.

### Antes de empezar
1. **GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU* (o L4/A100 si tienes Colab Pro).
2. **Licencia del modelo:** entra a la página del modelo que vayas a usar ([translategemma-4b-it](https://huggingface.co/google/translategemma-4b-it), [translategemma-12b-it](https://huggingface.co/google/translategemma-12b-it) o [gemma-4-26B-A4B-it](https://huggingface.co/google/gemma-4-26B-A4B-it)), inicia sesión y acepta las condiciones si las pide.
3. **Token de Hugging Face:** crea un token de lectura en [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) y guárdalo en Colab en el panel 🔑 *Secretos* con el nombre `HF_TOKEN` (activa el acceso para este notebook).
4. **Tu EPUB:** súbelo a Google Drive o, si no está ahí, la celda 2 te deja subirlo desde tu computadora.

### Uso
Ejecuta las celdas en orden. Para pausar, usa el botón ⏹ de la celda 6: el progreso queda guardado. Para seguir, vuelve a ejecutar las celdas 2 → 6.


## 1. Configuración


In [ ]:
#@title Configuración { display-mode: "form" }
# Ruta del EPUB dentro de Google Drive (si no existe, la celda 2 permite subirlo)
EPUB_PATH = "/content/drive/MyDrive/epub-translator/libro.epub"  #@param {type:"string"}
# Idioma del libro (obligatorio): código ISO, p. ej. de, en, fr, it, pt-BR
SRC_LANG = ""  #@param {type:"string"}
# Idioma destino: español, salvo que indiques otro
TGT_LANG = "es"  #@param {type:"string"}
# Modelo. Calidad medida en 52 párrafos de un libro alemán:
# - translategemma-4b-it: el más rápido; entra en cualquier GPU.
# - translategemma-12b-it: mejor vocabulario (en L4/T4 se carga en 4 bits).
# - gemma-4-26B-A4B-it (por defecto): la mejor calidad, sobre todo en modismos y
#   acepciones. Necesita una L4 o superior (en L4 se carga en 4 bits); con una
#   T4 elige translategemma-4b-it.
# Cambiar de modelo retraduce el libro (el modelo es parte de la clave de caché).
MODEL_ID = "google/gemma-4-26B-A4B-it"  #@param ["google/translategemma-4b-it", "google/translategemma-12b-it", "google/gemma-4-26B-A4B-it"] {allow-input: true}
# Cuantización: "auto" usa el modelo completo si entra en la GPU y, si no, 4 bits
# (8 bits ocupa menos que el modelo completo, pero genera mucho más lento).
QUANTIZATION = "auto"  #@param ["auto", "none", "8bit", "4bit"]
# Párrafos que se traducen a la vez. 8 es seguro en una T4 (15 GB); con L4/A100 prueba 16.
# Si la GPU se queda sin memoria, se reduce solo.
BATCH_SIZE = 8  #@param {type:"integer"}
# Carpeta de Drive donde se guardan la caché de traducciones y los EPUB generados
WORK_ROOT = "/content/drive/MyDrive/epub-translator/work"  #@param {type:"string"}
# Carpeta de Drive con las notas de traducción comunes a todos los libros
# (<src>-<tgt>.md, p. ej. de-es.md). Las notas de un solo libro van en
# <libro>.notas.md junto al EPUB. Si no existen, se traduce sin notas.
NOTES_DIR_PATH = "/content/drive/MyDrive/epub-translator/notas"  #@param {type:"string"}


## 2. Instalar dependencias, conectar Google Drive y cargar el EPUB


In [ ]:
!pip install -q -U "transformers>=4.56" accelerate bitsandbytes beautifulsoup4 lxml

import shutil
from pathlib import Path
from google.colab import drive, files

if not SRC_LANG.strip():
    raise SystemExit("Indica el idioma del libro en SRC_LANG (celda 1), p. ej. de o en.")

drive.mount("/content/drive")

EPUB = Path(EPUB_PATH)
if not EPUB.exists():
    print(f"No encontré {EPUB}. Elige el EPUB en tu computadora:")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No se subió ningún archivo.")
    name = next(iter(uploaded))
    EPUB.parent.mkdir(parents=True, exist_ok=True)
    EPUB = EPUB.parent / name
    shutil.move(name, EPUB)
print(f"EPUB: {EPUB} ({EPUB.stat().st_size / 1024:.0f} KB)")


## 3. Código de traducción y armado del EPUB

Esta celda solo define funciones (no usa la GPU). Es la misma lógica del script `translate_epub.py` / `build_epub.py`.

**Cómo funciona**
- La unidad de traducción es el **párrafo** (bloque sin sub-bloques): el modelo tiene la oración completa y la estructura HTML se conserva tal cual.
- Cursiva/negrita se envían como **Markdown**, que el modelo respeta mucho mejor que etiquetas HTML. Al volver, el Markdown se convierte otra vez en las etiquetas originales, aunque el modelo haya cambiado algún marcador: en el libro nunca quedan asteriscos.
- Todo se guarda en una **caché JSONL** append-only (`cache_<src>-<tgt>.jsonl`). La clave es un hash de modelo + idiomas + texto: lo repetido se traduce una vez y cambiar de modelo no mezcla resultados.
- Para saber qué falta, se ejecuta el **mismo código que aplica las traducciones**, anotando lo que no está en la caché. Así lo que se pide y lo que se busca siempre coincide.
- **Notas de traducción:** en `NOTES_DIR_PATH/<src>-<tgt>.md` (celda 1) puedes explicarle al modelo qué acepción usar para ciertas palabras. Cada nota empieza con `## <patrón>` y sigue con la explicación; solo se agrega al prompt de los párrafos donde aparece el patrón, y al cambiarla solo se retraducen esos párrafos.


In [ ]:
import gc
import hashlib
import json
import posixpath
import re
import shutil
import time
import zipfile
from collections import deque
from pathlib import Path
from typing import Callable, NamedTuple
from urllib.parse import unquote

from bs4 import BeautifulSoup, NavigableString, Tag
from lxml import etree

# La caché usa MODEL_NAME en su clave: un modelo distinto no reutiliza traducciones
MODEL_NAME = MODEL_ID

# ─── Constantes ────────────────────────────────────────────────────────────────

# max_tokens de salida = tokens de entrada * factor + margen
MAX_TOKENS_FACTOR = 2.5


MAX_TOKENS_MARGIN = 64


# Salida sospechosa si supera len(origen) * factor + margen (el modelo "explica"
# en vez de traducir, típico en fragmentos cortos sin contexto)
MAX_OUTPUT_RATIO = 3


MAX_OUTPUT_MARGIN = 50


END_OF_TURN = "<end_of_turn>"


HTML_EXTENSIONS = {".html", ".xhtml", ".htm"}


SKIP_TAGS = {"style", "script", "head", "title", "link", "meta", "svg", "math"}


BLOCK_TAGS = {
    "p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "dt", "dd", "td", "th",
    "caption", "blockquote", "div", "figcaption", "pre", "section", "article",
    "aside", "header", "footer", "nav", "ul", "ol", "dl", "table", "tr",
    "tbody", "thead", "tfoot", "figure", "body", "html",
}


INLINE_TAGS = {
    "a", "abbr", "b", "bdi", "bdo", "cite", "code", "dfn", "em", "i", "kbd",
    "mark", "q", "s", "samp", "small", "span", "strong", "sub", "sup", "time",
    "u", "var", "font", "del", "ins",
}


# lookup(texto) -> traducción o None. Es la única forma en que el código de HTML
# accede a las traducciones: según qué lookup se le pase, el mismo código sirve
# para planificar (anotar lo que falta) o para aplicar.
Lookup = Callable[[str], "str | None"]


# Inline que se traduce como Markdown (y se reconstruye)
MD_TAGS = {"em": "*", "i": "*", "cite": "*", "dfn": "*", "var": "*",
           "b": "**", "strong": "**"}


# Inline que obliga a traducir fragmento por fragmento para no perderlo
# (links, notas al pie, saltos de línea e imágenes no se pueden expresar en
# Markdown de forma que el modelo los respete)
PRESERVE_TAGS = {"a", "sup", "sub", "br", "img", "wbr", "code", "kbd", "samp"}
# Elementos sin texto que igual se ven: no son "anclas" que se puedan mover
VISIBLE_EMPTY_TAGS = ["br", "img", "wbr", "hr", "svg", "image", "math"]

# Párrafos pendientes que se toman por "ventana" (en orden de lectura). Dentro de
# la ventana se ordenan por largo para armar lotes parejos: con batching estático
# (transformers) cada lote espera a su párrafo más largo y rellena (padding) los
# demás, así que agrupar largos parecidos evita cómputo desperdiciado. La ventana
# acotada mantiene el orden de lectura a grandes rasgos, que es lo que necesita
# el EPUB parcial.
WINDOW_BATCHES = 8
# Penalización de repetición solo en el reintento (rompe bucles "die die die...")
RETRY_REPETITION_PENALTY = 1.2


# ─── Notas de traducción (acepciones, modismos) ────────────────────────────────
#
# Archivos Markdown donde cada nota empieza con "## <patrón>" y sigue con la
# explicación en lenguaje natural:
#
#     ## knips
#     «knipsen» y sus formas significan siempre sacar fotos: tradúcelo como
#     «fotografiar» o «sacar/tomar fotos». Nunca como golpear, jugar ni mirar.
#
# El patrón es una expresión regular (sin distinguir mayúsculas) que se busca en
# el texto ORIGINAL. Cada nota se agrega al prompt solo de los textos donde
# aparece su patrón: el prompt queda corto y el resto del libro no cambia.
#
# Por qué funciona: TranslateGemma sigue instrucciones de terminología puestas
# en su prompt. Probado con "knipsen" en 10 párrafos mal traducidos: sin nota, 2
# usaban la acepción correcta; con la nota, 10, sin que el texto de la nota se
# colara en la traducción.
#
# Se combinan dos archivos:
# - <NOTES_DIR_PATH>/<src>-<tgt>.md en Drive: notas comunes a todos los libros
#   de ese par de idiomas (modismos, usos que se repiten);
# - <libro>.notas.md junto al EPUB: notas propias de un libro (nombres, jerga).

NOTES_DIR = Path(NOTES_DIR_PATH)  # celda 1
# El encabezado del bloque de notas va en el idioma destino cuando se conoce
# (probado: en español da resultados algo mejores que en inglés para de → es).
NOTES_HEADER = {"es": "Terminología (respétala siempre):"}
NOTES_HEADER_DEFAULT = "Terminology (follow strictly):"
# Las notas se insertan en el prompt de TranslateGemma justo antes de esta frase
# de su plantilla, es decir, después de su descripción del rol de traductor.
PROMPT_NOTES_ANCHOR = "Produce only the"


class Note(NamedTuple):
    pattern: re.Pattern
    text: str
    origin: str  # "archivo:línea", para mensajes


def parse_notes(path: Path) -> list[Note]:
    """Lee un archivo de notas. El texto antes del primer "## " se ignora
    (sirve para explicar el archivo)."""
    notes: list[Note] = []
    current: tuple[re.Pattern, list[str], str] | None = None

    def close():
        if current:
            body = " ".join(" ".join(current[1]).split())
            if body:
                notes.append(Note(current[0], body, current[2]))

    for n, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if line.startswith("## "):
            close()
            raw = line[3:].strip()
            try:
                current = (re.compile(raw, re.IGNORECASE), [], f"{path.name}:{n}")
            except re.error as e:
                raise SystemExit(f"❌ {path}:{n}: patrón inválido {raw!r}: {e}")
        elif current is not None:
            current[1].append(line)
    close()
    return notes


def notes_files(input_epub: Path, src_lang: str, tgt_lang: str) -> list[Path]:
    """Archivos de notas que existen para este libro y par de idiomas."""
    candidates = [NOTES_DIR / f"{src_lang}-{tgt_lang}.md",
                  input_epub.with_name(f"{input_epub.stem}.notas.md")]
    return [p for p in candidates if p.exists()]


def load_notes(input_epub: Path, src_lang: str, tgt_lang: str) -> list[Note]:
    return [n for p in notes_files(input_epub, src_lang, tgt_lang) for n in parse_notes(p)]


def notes_block(notes: list[Note], text: str, tgt_lang: str) -> str:
    """Bloque de notas para el prompt de `text`, o "" si ninguna aplica."""
    hits = [n for n in notes if n.pattern.search(text)]
    if not hits:
        return ""
    header = NOTES_HEADER.get(tgt_lang.split("-")[0].lower(), NOTES_HEADER_DEFAULT)
    return header + "\n" + "".join(f"- {n.text}\n" for n in hits)


def insert_notes(prompt: str, block: str) -> str:
    """Inserta el bloque de notas en el prompt ya renderizado por la plantilla."""
    i = prompt.find(PROMPT_NOTES_ANCHOR)
    if i < 0:
        raise RuntimeError(f"No encontré {PROMPT_NOTES_ANCHOR!r} en el prompt del modelo: "
                           "las notas de traducción no son compatibles con esta plantilla.")
    return prompt[:i] + block + prompt[i:]


# ─── Modelo: tipo y prompt ───────────────────────────────────────────────────────

# Tipo de modelo: TranslateGemma usa su propia plantilla (idiomas como campos);
# un modelo general ("chat") recibe instrucciones de traductor como mensaje de sistema.
MODEL_KIND = "translategemma" if "translategemma" in MODEL_ID.lower() else "chat"

END_MARKERS_RE = re.compile(r"<end_of_turn>|<turn\|>")

THINKING_RE = re.compile(r"<\|channel>.*?<channel\|>", re.DOTALL)

# Instrucciones para modelos generales ("chat")
CHAT_SYSTEM_PROMPT = (
    "You are a professional literary translator from {src} to {tgt}. Translate the "
    "user's text into {tgt}. Preserve meaning, tone and register; translate colloquialisms "
    "and idioms by their meaning, not word for word. Keep the Markdown emphasis markers "
    "(*x*, **x**) around the corresponding words, and keep the original quotation marks "
    "and punctuation style. Reply only with the translation, without comments, notes or "
    "surrounding quotes.")

LANG_NAMES = {"de": "German", "en": "English", "es": "Spanish", "fr": "French",
              "it": "Italian", "pt": "Portuguese", "nl": "Dutch", "pl": "Polish",
              "ru": "Russian", "ca": "Catalan", "sv": "Swedish", "da": "Danish"}

def lang_name(code: str) -> str:
    return LANG_NAMES.get(code.split("-")[0].split("_")[0].lower(), code)

def clean_output(text: str) -> str:
    """Quita marcas de control de la salida del modelo."""
    return END_MARKERS_RE.split(THINKING_RE.sub("", text))[0].strip()


# ─── Caché y utilidades de texto ───────────────────────────────────────────────

class TranslationCache:
    """Caché append-only en JSONL: una línea {"k": hash, "t": traducción} por texto.

    Es el único estado persistente de la traducción:
    - Append + flush por entrada: una interrupción pierde como mucho las
      secuencias que se estaban generando, nunca lo ya traducido.
    - Si un texto se vuelve a traducir (p. ej. para corregir una salida mala),
      la línea nueva se agrega al final y, al leer, la última gana.
    - Un texto que el modelo no logró traducir se guarda con "f": 1 y el texto
      original: el libro se puede armar igual (ese párrafo queda sin traducir),
      pero la celda 6 lo vuelve a intentar en la próxima ejecución.
    - read_only=True permite armar el EPUB (celda 7) mientras la traducción
      sigue escribiendo: solo se agregan líneas, nunca se reescribe el archivo.
    """

    def __init__(self, path: Path, src_lang: str, tgt_lang: str, read_only: bool = False,
                 notes: list[Note] | None = None):
        self.path = path
        self.prefix = f"{MODEL_NAME}|{src_lang}>{tgt_lang}\n"
        self.tgt_lang = tgt_lang
        self.notes = notes or []
        self.data: dict[str, str] = {}
        self.failed_keys: set[str] = set()  # claves guardadas con el original
        if path.exists():
            with path.open(encoding="utf-8") as f:
                for line in f:
                    try:
                        entry = json.loads(line)
                        self.data[entry["k"]] = entry["t"]
                    except (json.JSONDecodeError, KeyError, TypeError):
                        continue  # línea truncada por una interrupción
                    # La última línea de cada clave gana, también para la marca
                    if entry.get("f"):
                        self.failed_keys.add(entry["k"])
                    else:
                        self.failed_keys.discard(entry["k"])
        self._fh = None
        if read_only:
            return
        self._fh = path.open("a", encoding="utf-8")
        if path.stat().st_size and not path.read_bytes().endswith(b"\n"):
            self._fh.write("\n")  # cerrar línea truncada

    def notes_block(self, text: str) -> str:
        """Notas que aplican a `text` (las que van en su prompt), o ""."""
        return notes_block(self.notes, text, self.tgt_lang)

    def _key(self, text: str) -> str:
        # Incluir modelo e idiomas en la clave evita reutilizar traducciones de
        # otra configuración; el hash mantiene las claves cortas y de largo fijo.
        # Las notas que aplican al texto también son parte de la clave: si se
        # agrega o cambia una nota, solo se retraducen los textos que la usan.
        # Sin notas el bloque es "" y la clave es la de siempre.
        return hashlib.sha1((self.prefix + self.notes_block(text) + text)
                            .encode("utf-8")).hexdigest()

    def get(self, text: str) -> str | None:
        return self.data.get(self._key(text))

    def is_failed(self, text: str) -> bool:
        """¿Lo guardado para `text` es el original porque la traducción falló?"""
        return self._key(text) in self.failed_keys

    def put(self, text: str, translated: str, failed: bool = False):
        key = self._key(text)
        self.data[key] = translated
        entry = {"k": key, "t": translated}
        if failed:
            entry["f"] = 1
            self.failed_keys.add(key)
        else:
            self.failed_keys.discard(key)
        self._fh.write(json.dumps(entry, ensure_ascii=False) + "\n")
        # flush por entrada: el costo es despreciable al lado de la generación
        # (~0.4 s por texto) y garantiza que el progreso quede en disco.
        self._fh.flush()

    def close(self):
        if self._fh:
            self._fh.close()


def split_ws(text: str) -> tuple[str, str, str]:
    """Separa (espacio inicial, núcleo, espacio final)."""
    core = text.strip()
    if not core:
        return text, "", ""
    start = text.index(core)
    return text[:start], core, text[start + len(core):]


def has_letters(text: str) -> bool:
    return any(c.isalpha() for c in text)


def output_problem(src: str, out: str) -> str | None:
    """Motivo por el que una traducción parece mala, o None si parece correcta.

    Heurísticas baratas para los fallos reales observados:
    - la misma palabra 8+ veces seguidas: bucle de la decodificación greedy;
    - salida > 3x el original + 50: el modelo "explica" en vez de traducir
      (pasa con fragmentos cortos como "th" o "WiE ES"). En el libro de prueba
      ninguna traducción normal de más de 40 caracteres superó 1.9x; el margen
      fijo evita falsos positivos en textos muy cortos.
    Se usa al generar y también al leer la caché, para corregir entradas malas
    de ejecuciones anteriores. Sobre las 3237 traducciones del libro de prueba
    marcó solo los 3 casos malos reales.
    """
    if not out:
        return "salida vacía"
    if re.search(r"\b(\w+)(?:\W+\1\b){7,}", out):
        return "bucle de repetición"
    if len(out) > len(src) * MAX_OUTPUT_RATIO + MAX_OUTPUT_MARGIN:
        return "salida demasiado larga"
    return None


def is_skipped(node) -> bool:
    return any(p.name in SKIP_TAGS for p in node.parents if p.name)


# ─── Unidades de traducción ────────────────────────────────────────────────────
#
# Estrategia por bloque (párrafo, título, celda...), de mejor a peor:
#   1. "md":    cursiva/negrita como Markdown (*x*, **x**); la salida se vuelve a
#               convertir en las etiquetas originales (md_to_nodes).
#   2. "plain": bloques sin cursiva/negrita: el texto entero. Si el modelo agrega
#               Markdown por su cuenta, también se convierte en etiquetas.
#   3. "nodes": bloques con links, notas, <br>, imágenes o ids con texto (las
#               anclas vacías, como los marcadores de página, se reinsertan):
#               cada fragmento de texto por separado, para no perder nada.

def collect_units(soup) -> list:
    """Devuelve unidades a traducir en orden de documento:
    - Tag: un bloque hoja (párrafo, título, celda...): bloque sin sub-bloques
    - NavigableString: texto suelto fuera de un bloque hoja

    Recorrido recursivo único, O(n): cada llamada devuelve si su subárbol
    contiene un bloque, así el padre sabe si es "hoja" sin volver a recorrer
    sus descendientes (hacerlo por cada bloque sería O(n²) en HTML anidado).
    """
    def walk(node, skipped: bool) -> tuple[bool, list]:
        """Devuelve (contiene un bloque, unidades)."""
        has_block, found = False, []
        for child in node.children:
            if isinstance(child, Tag):
                child_block, child_units = walk(child, skipped or child.name in SKIP_TAGS)
                has_block |= child_block or child.name in BLOCK_TAGS
                found.extend(child_units)
            elif type(child) is NavigableString and not skipped and has_letters(child):
                found.append(child)
        if (isinstance(node, Tag) and node.name in BLOCK_TAGS
                and not has_block and not skipped):
            return False, [node] if has_letters(node.get_text()) else []
        return has_block, found

    return walk(soup, False)[1]


def unwrap_target(block: Tag) -> Tag:
    """Si el bloque solo envuelve un elemento inline (<h4><span>17</span></h4>,
    <p><a href>Kapitel</a></p>), traduce el contenido de ese elemento.

    Así un envoltorio que abarca todo el bloque (un link del índice, un span
    de estilo) se conserva intacto y no obliga a usar el modo "nodes"."""
    target = block
    while True:
        sig = [c for c in target.children
               if not (type(c) is NavigableString and not c.strip())]
        if len(sig) == 1 and isinstance(sig[0], Tag) and sig[0].name in INLINE_TAGS:
            target = sig[0]
        else:
            return target


def is_anchor(tag: Tag) -> bool:
    """Elemento sin texto que solo marca una posición: marcador de página
    (<span epub:type="pagebreak" id="page-18"/>), destino de un link (<a id>)...
    No cuenta br/img/wbr, que sí cambian lo que se ve."""
    return (tag.name not in VISIBLE_EMPTY_TAGS and not tag.get_text().strip()
            and tag.find(VISIBLE_EMPTY_TAGS) is None)


def anchors_in(target: Tag) -> list[tuple[Tag, float]]:
    """Anclas de primer nivel del bloque y su posición relativa (0-1) en el texto."""
    total = sum(len(s) for s in target.find_all(string=True)) or 1
    found, pos = [], 0
    for d in target.descendants:
        if isinstance(d, NavigableString):
            pos += len(d)
        elif is_anchor(d) and not any(is_anchor(p) for p in d.parents
                                      if p is not target and target in p.parents):
            found.append((d, pos / total))
    return found


def insert_at(target: Tag, element: Tag, ratio: float):
    """Inserta `element` en la posición relativa `ratio` del texto de `target`,
    al inicio de la palabra más cercana (nunca en medio de una palabra)."""
    strings = [s for s in target.find_all(string=True) if type(s) is NavigableString]
    total = sum(len(s) for s in strings)
    goal, acc = round(ratio * total), 0
    if goal == 0:  # al inicio del bloque, no dentro de su primer hijo
        target.insert(0, element)
        return
    for s in strings:
        text = str(s)
        if acc + len(text) >= goal:
            k = goal - acc
            if 0 < k < len(text) and not text[k - 1].isspace():
                nxt = re.search(r"\s", text[k:])
                k = k + nxt.end() if nxt else len(text)
            parts = [NavigableString(text[:k]), element, NavigableString(text[k:])]
            s.replace_with(*[x for x in parts if not isinstance(x, NavigableString) or x])
            return
        acc += len(text)
    target.append(element)


def block_mode(target: Tag) -> str:
    """Elige la estrategia del bloque: "md", "plain" o "nodes" (ver arriba).

    "nodes" se reserva para lo que no se puede perder (links, notas, anclas
    con texto), porque traduce peor: cada fragmento llega al modelo sin
    contexto. Las anclas vacías (marcadores de página) no cuentan: se sacan
    antes de traducir y se reinsertan después en la posición equivalente. Un
    texto que ya tiene "*" no puede usar Markdown (sería ambiguo)."""
    anchors = {id(a) for a in target.find_all(True) if is_anchor(a)}
    for t in target.find_all(True):
        if id(t) in anchors or any(id(p) in anchors for p in t.parents):
            continue
        if t.name not in INLINE_TAGS or t.name in PRESERVE_TAGS or t.has_attr("id"):
            return "nodes"
    if any(type(c) is not NavigableString for c in target.find_all(string=True)):
        return "nodes"  # comentarios, CDATA...
    if any(t.name in MD_TAGS for t in target.find_all(True)):
        return "plain" if "*" in target.get_text() else "md"
    return "plain"


def md_elements(target: Tag) -> list[Tag]:
    return [t for t in target.find_all(True)
            if t.name in MD_TAGS and t.get_text().strip()]


def encode_md(target: Tag) -> str:
    """Texto del bloque con cursiva/negrita como *x* / **x**.

    Los espacios del borde del elemento quedan fuera de los marcadores
    ("* Hotel*" no es Markdown válido y el modelo lo rompería). Los inline
    que no son de MD_TAGS (span, u, ...) se vuelven transparentes: su texto
    se traduce y restore_inline() intenta recuperar el elemento después."""
    def walk(node) -> str:
        parts = []
        for child in node.children:
            if isinstance(child, Tag):
                inner = walk(child)
                if child.name in MD_TAGS and child.get_text().strip():
                    lead, core, trail = split_ws(inner)
                    mark = MD_TAGS[child.name]
                    parts.append(f"{lead}{mark}{core}{mark}{trail}")
                else:
                    parts.append(inner)  # inline transparente (span, u, ...)
            else:
                parts.append(str(child))
        return "".join(parts)
    return walk(target)


MD_RE = re.compile(r"\*\*|\*")


def parse_md(text: str) -> list | None:
    """Árbol [texto | (marca, hijos)] si los * de `text` están bien cerrados,
    o None si no (un * suelto o cierres cruzados como *a **b* c**)."""
    root: list = []
    stack: list[tuple[str, list]] = [("", root)]
    pos = 0
    for m in MD_RE.finditer(text):
        if text[pos:m.start()]:
            stack[-1][1].append(text[pos:m.start()])
        pos = m.end()
        mark = m.group(0)
        if stack[-1][0] == mark:
            stack[-2][1].append(stack.pop())
        elif any(open_mark == mark for open_mark, _ in stack[1:]):
            return None
        else:
            stack.append((mark, []))
    if text[pos:]:
        stack[-1][1].append(text[pos:])
    return root if len(stack) == 1 else None


def md_to_nodes(text: str, soup, elements: list[Tag]) -> tuple[list, set[int]]:
    """Convierte la salida del modelo (texto con Markdown) en nodos HTML.
    Devuelve (nodos, ids de los elementos de `elements` reutilizados).

    Tolerante a propósito: el modelo a menudo agrega, quita o mueve algún
    marcador (en el libro de prueba, en ~40% de los párrafos con cursiva).
    Exigir que coincidan exactamente con el original hacía descartar todo el
    formato del párrafo, y aceptar el texto tal cual dejaba asteriscos en el
    libro. En cambio:
    - Cada tramo *x* / **x** usa el siguiente elemento original del mismo tipo
      (copiando nombre y atributos, p. ej. <em class="titulo">); si ya no
      quedan, un <em>/<strong> genérico (el modelo puso en cursiva algo que
      el original marcaba de otra forma, como un título entre ›comillas‹).
    - Si los marcadores no están bien cerrados, se eliminan: se pierde ese
      formato, nunca el texto, y no queda ningún asterisco suelto.
    Los elementos originales que no se reutilizaron los puede recuperar
    restore_inline() si su texto aparece literal en la traducción."""
    tree = parse_md(text)
    if tree is None:
        return [NavigableString(MD_RE.sub("", text))], set()
    pools = {mark: deque(e for e in elements if MD_TAGS[e.name] == mark)
             for mark in set(MD_TAGS.values())}
    used: set[int] = set()

    def plain_text(items) -> str:
        return "".join(it if isinstance(it, str) else plain_text(it[1]) for it in items)

    def build(items) -> list:
        out = []
        for it in items:
            if isinstance(it, str):
                out.append(NavigableString(it))
                continue
            mark, children = it
            inner = build(children)
            if not plain_text(children).strip():
                out.extend(inner)  # "**" vacío: no crear una etiqueta vacía
                continue
            if pools[mark]:
                src = pools[mark].popleft()
                used.add(id(src))
                tag = soup.new_tag(src.name, attrs=dict(src.attrs))
            else:
                tag = soup.new_tag("em" if mark == "*" else "strong")
            for n in inner:
                tag.append(n)
            out.append(tag)
        return out

    return build(tree), used


def leaf_inline(target: Tag) -> list[Tag]:
    """Elementos inline sin hijos y con texto: los candidatos a restore_inline()."""
    return [t for t in target.find_all(True) if not t.find(True) and t.get_text().strip()]


def inside(node, name: str, stop: Tag) -> bool:
    """¿`node` está dentro de un elemento `name`, sin pasar de `stop`?"""
    for p in node.parents:
        if p is stop:
            return False
        if p.name == name:
            return True
    return False


def restore_inline(target: Tag, originals: list[tuple[str, dict, str]], soup):
    """Re-aplica formato inline perdido cuando su texto aparece literal en la
    traducción (siglas en versalitas, números, nombres propios en cursiva...).

    Funciona porque lo que suele llevar formato tipográfico (siglas, cifras,
    títulos de obras) casi nunca se traduce: si el texto exacto aparece en la
    salida, es seguro volver a envolverlo. Si no aparece, se pierde solo el
    estilo, nunca el texto. No envuelve texto que ya está dentro de un
    elemento del mismo tipo (evita <em><em>x</em></em>)."""
    restored: set[int] = set()
    for name, attrs, text in originals:
        pat = re.compile(r"(?<!\w)" + re.escape(text.strip()) + r"(?!\w)")
        for node in target.find_all(string=True):
            if type(node) is not NavigableString or id(node.parent) in restored:
                continue
            if inside(node, name, target):
                continue
            m = pat.search(str(node))
            if not m:
                continue
            new = soup.new_tag(name, attrs=attrs)
            new.string = m.group(0)
            parts = [NavigableString(str(node)[:m.start()]), new,
                     NavigableString(str(node)[m.end():])]
            node.replace_with(*[x for x in parts if not isinstance(x, NavigableString) or x])
            restored.add(id(new))
            break


def fill_block(target: Tag, text: str, soup, source_has_stars: bool):
    """Reemplaza el contenido de `target` por la traducción `text`.

    Si el original no tenía asteriscos, los de la salida son Markdown (nuestro
    o agregado por el modelo) y se convierten en etiquetas; si los tenía (p. ej.
    marcas de notas al pie), se dejan como texto."""
    elements = md_elements(target)
    originals = leaf_inline(target)
    anchors = anchors_in(target)
    for anchor, _ in anchors:
        anchor.extract()
    if source_has_stars:
        nodes, used = [NavigableString(text)], set()
    else:
        nodes, used = md_to_nodes(text, soup, elements)
    target.clear()
    for n in nodes:
        target.append(n)
    restore_inline(target, [(t.name, dict(t.attrs), t.get_text())
                            for t in originals if id(t) not in used], soup)
    # Marcadores de página y anclas: en la misma posición relativa del texto.
    # Sin esto, un párrafo con un marcador de página en medio se traducía por
    # fragmentos: oraciones partidas y palabras pegadas (en un libro hecho
    # desde PDF, 23% de los párrafos tenían uno).
    for anchor, ratio in anchors:
        insert_at(target, anchor, ratio)


def keep_edge_punctuation(source: str, result: str) -> str:
    """Repone la puntuación de los bordes de un fragmento si el modelo la quitó.

    Al traducir un fragmento suelto (modo "nodes"), el modelo tiende a omitir
    la puntuación inicial: ", such as the family" → "como la familia". Sin la
    coma ni el espacio, la palabra anterior (p. ej. una cursiva) quedaba pegada
    ("ministerialescomo"). También repone comas, dos puntos o signos de
    apertura al final, que unen el fragmento con el siguiente."""
    lead = re.match(r"[,.;:!?)\]»”’]+\s*", source)
    if lead and not result.startswith(lead.group(0).rstrip()[0]):
        result = lead.group(0) + result.lstrip()
    trail = re.search(r"\s*[,;:(\[«“‘]$", source)
    if trail and not result.endswith(trail.group(0).strip()):
        result = result.rstrip() + trail.group(0)
    return result


def translate_string(node: NavigableString, lookup: Lookup, soup=None) -> bool:
    # Solo el núcleo va al modelo (que recorta espacios); los espacios del borde
    # se reponen, así no se pegan palabras con la etiqueta vecina.
    lead, core, trail = split_ws(str(node))
    result = lookup(core)
    if result is None:
        return False
    text = lead + keep_edge_punctuation(core, result) + trail
    if "*" in str(node) or not MD_RE.search(result):
        node.replace_with(NavigableString(text))
    elif soup is None or any(p.name in MD_TAGS for p in node.parents if p.name):
        # Ya está dentro de una cursiva/negrita: el Markdown del modelo sobra
        node.replace_with(NavigableString(MD_RE.sub("", text)))
    else:
        node.replace_with(*md_to_nodes(text, soup, [])[0])
    return True


def translate_block(block: Tag, lookup: Lookup, soup) -> bool:
    """Aplica la traducción a un bloque. `lookup(texto)` devuelve la traducción
    o None si todavía no existe (en ese caso el bloque queda sin cambios).
    Devuelve True si el bloque quedó completamente traducido."""
    target = unwrap_target(block)
    mode = block_mode(target)
    if mode in ("md", "plain"):
        source = encode_md(target) if mode == "md" else target.get_text()
        lead, core, trail = split_ws(source)
        result = lookup(core)
        if result is None:
            return False
        fill_block(target, lead + result + trail, soup,
                   source_has_stars="*" in target.get_text())
        return True
    ok = True
    for s in [s for s in target.find_all(string=True)
              if type(s) is NavigableString and has_letters(s) and not is_skipped(s)]:
        ok &= translate_string(s, lookup, soup)
    return ok


# ─── Estructura del EPUB ───────────────────────────────────────────────────────

def book_dirs(input_epub: Path) -> tuple[Path, Path]:
    """(work_dir, src_dir) del libro, dentro de WORK_ROOT (Google Drive)."""
    work_dir = Path(WORK_ROOT) / input_epub.stem
    return work_dir, work_dir / "src"


def cache_path(work_dir: Path, src_lang: str, tgt_lang: str) -> Path:
    return work_dir / f"cache_{src_lang}-{tgt_lang}.jsonl"


def extract_epub(epub_path: Path, src_dir: Path):
    """Descomprime el EPUB una sola vez. src/ nunca se modifica: es la fuente
    de verdad tanto para planificar la traducción como para armar el EPUB."""
    if (src_dir / "mimetype").exists():
        print(f"[extract] Ya existe en {src_dir}, skip.")
        return
    src_dir.mkdir(parents=True, exist_ok=True)
    print(f"[extract] {epub_path.name} → {src_dir}")
    with zipfile.ZipFile(epub_path) as zf:
        zf.extractall(src_dir)
    count = sum(1 for f in src_dir.rglob("*") if f.is_file())
    print(f"[extract] {count} archivos.")


def find_opf(src_dir: Path) -> Path:
    container = etree.parse(str(src_dir / "META-INF" / "container.xml"))
    full_path = container.find(".//{*}rootfile").get("full-path")
    return src_dir / unquote(full_path)


def resolve_href(base_file: Path, href: str) -> Path:
    """Ruta absoluta de un href relativo a `base_file` (sin #fragmento)."""
    return Path(posixpath.normpath(str(base_file.parent / unquote(href.split("#")[0]))))


def spine_files(src_dir: Path, include_rest: bool = True) -> list[Path]:
    """HTML en orden de lectura (spine); al final, los HTML fuera del spine.

    El orden importa: traducir siguiendo el spine hace que
    los capítulos se completen de principio a fin, que es lo que necesita un
    EPUB parcial. El orden alfabético de archivos no sirve ("chapter10" va
    antes que "chapter2")."""
    opf = find_opf(src_dir)
    tree = etree.parse(str(opf))
    hrefs = {it.get("id"): it.get("href") for it in tree.iterfind(".//{*}manifest/{*}item")}
    ordered: list[Path] = []
    for ref in tree.iterfind(".//{*}spine/{*}itemref"):
        href = hrefs.get(ref.get("idref"))
        if href:
            path = resolve_href(opf, href)
            if path.suffix.lower() in HTML_EXTENSIONS and path.exists() and path not in ordered:
                ordered.append(path)
    if not include_rest:
        return ordered
    rest = sorted(f for f in src_dir.rglob("*")
                  if f.is_file() and f.suffix.lower() in HTML_EXTENSIONS and f not in ordered)
    return ordered + rest


def find_ncx(src_dir: Path) -> Path | None:
    opf = find_opf(src_dir)
    tree = etree.parse(str(opf))
    for it in tree.iterfind(".//{*}manifest/{*}item"):
        if it.get("media-type") == "application/x-dtbncx+xml":
            return resolve_href(opf, it.get("href"))
    return None


def ncx_label_nodes(ncx_tree) -> list:
    """Elementos <text> de títulos del índice NCX (docTitle y navLabel)."""
    return [t for t in ncx_tree.iterfind(".//{*}text")
            if t.text and has_letters(t.text)]


def opf_title_texts(src_dir: Path) -> list[str]:
    """Títulos del libro en el OPF (<dc:title>). Se piden aparte porque no
    están en ningún HTML: en un EPUB3 sin índice NCX nadie más los pediría y
    la celda 7 dejaría el título en el idioma original."""
    tree = etree.parse(str(find_opf(src_dir)))
    return [split_ws(t.text)[1] for t in tree.iterfind(".//{*}metadata/{*}title")
            if t.text and has_letters(t.text)]


def apply_translations(html_file: Path, lookup: Lookup):
    """Parsea un HTML y aplica las traducciones disponibles.
    Devuelve (soup, texto_original, unidades, unidades_incompletas).

    Lo usan las dos fases: translate_book() para planificar (con un lookup que anota lo que
    falta) y build() para armar el libro (con cache.get). Compartir el
    código garantiza que las claves pedidas y las buscadas coincidan."""
    text = html_file.read_text(encoding="utf-8")
    soup = BeautifulSoup(text, "html.parser")
    units = collect_units(soup)
    incomplete = 0
    for u in units:
        ok = (translate_block(u, lookup, soup) if isinstance(u, Tag)
              else translate_string(u, lookup, soup))
        incomplete += not ok
    return soup, text, len(units), incomplete


# ─── Bucle de traducción ───────────────────────────────────────────────────────

def is_oom(e: Exception) -> bool:
    return type(e).__name__ == "OutOfMemoryError" or "out of memory" in str(e).lower()


def translate_pending(tr, cache: TranslationCache, texts: list[str],
                      batch_size: int = 8, failed: set[str] | None = None) -> int:
    """Traduce `texts` (en orden de lectura) por lotes y guarda cada resultado en
    la caché apenas termina su lote. Devuelve el tamaño de lote final.

    - Ventanas de WINDOW_BATCHES lotes, ordenadas por largo (menos padding).
    - Salida mala (bucle, cortada, demasiado larga): un reintento con
      penalización de repetición y el doble de max_tokens; si vuelve a fallar
      se guarda el original marcado como fallo (se reintenta al relanzar).
    - Sin memoria en la GPU: se reduce el lote a la mitad y se repite.
    """
    queue = deque(texts)
    total, done, kept_original = len(texts), 0, 0
    start = last_report = time.time()
    bs = batch_size
    while queue:
        window = [queue.popleft() for _ in range(min(len(queue), bs * WINDOW_BATCHES))]
        retries: list[str] = []
        for attempt in (0, 1):
            items = sorted(window if attempt == 0 else retries, key=tr.n_tokens, reverse=True)
            i = 0
            while i < len(items):
                batch = items[i:i + bs]
                try:
                    outputs = tr.translate_batch(batch, [cache.notes_block(t) for t in batch],
                                                  retry=bool(attempt))
                except Exception as e:
                    if is_oom(e) and bs > 1:
                        bs = max(1, bs // 2)
                        tr.free_memory()
                        print(f"    ⚠ GPU sin memoria: lote reducido a {bs}", flush=True)
                        continue
                    raise
                for text, (result, finished) in zip(batch, outputs):
                    problem = ("cortada por max_tokens" if not finished
                               else output_problem(text, result))
                    if problem and attempt == 0:
                        print(f"    ⟳ Reintento ({problem}): {text[:60]!r}...", flush=True)
                        retries.append(text)
                        continue
                    if problem:
                        print(f"    ⚠ Se conserva el original ({problem}): {text[:60]!r}...",
                              flush=True)
                        result = text
                        kept_original += 1
                        if failed is not None:
                            failed.add(text)
                    cache.put(text, result, failed=bool(problem))
                    done += 1
                i += len(batch)
                now = time.time()
                if now - last_report >= 30 or not queue and i >= len(items):
                    last_report = now
                    rate = done / (now - start)
                    eta = (total - done) / rate if rate else 0
                    print(f"      {done}/{total} textos  ({rate * 60:.0f}/min, "
                          f"ETA ~{eta / 60:.0f} min)", flush=True)
    if kept_original:
        print(f"    ⚠ {kept_original} textos quedaron en el idioma original (ver arriba); "
              "se reintentan al volver a ejecutar la celda")
    return bs


def translate_book(tr, input_epub: Path, src_lang: str, tgt_lang: str, batch_size: int):
    """Extrae el EPUB (una vez) y traduce a la caché todo lo que falte."""
    work_dir, src_dir = book_dirs(input_epub)
    extract_epub(input_epub, src_dir)
    notes = load_notes(input_epub, src_lang, tgt_lang)
    files = notes_files(input_epub, src_lang, tgt_lang)
    print(f"[notes] {len(notes)} notas de " + ", ".join(f.name for f in files) if files
          else f"[notes] sin notas (puedes crear {NOTES_DIR}/{src_lang}-{tgt_lang}.md)")
    cache = TranslationCache(cache_path(work_dir, src_lang, tgt_lang), src_lang, tgt_lang,
                             notes=notes)
    html_files = spine_files(src_dir)
    ncx = find_ncx(src_dir)
    ncx_texts = ([split_ws(t.text)[1] for t in ncx_label_nodes(etree.parse(str(ncx)))]
                 if ncx and ncx.exists() else [])
    title_texts = opf_title_texts(src_dir)

    start = time.time()
    failed: set[str] = set()  # fallaron en esta ejecución: no se piden en cada ronda
    try:
        for round_no in range(1, 4):
            missing: dict[str, None] = {}  # dict = conjunto que conserva el orden

            def lookup(text: str) -> str | None:
                result = cache.get(text)
                if result is not None and output_problem(text, result):
                    result = None  # traducción mala de una ejecución anterior: se rehace
                # Los que fallaron en una ejecución anterior (guardados con el
                # original) se piden de nuevo, una vez por ejecución.
                if (result is None or cache.is_failed(text)) and text not in failed:
                    missing[text] = None
                return result

            for t in title_texts + ncx_texts:  # título e índice primero
                lookup(t)
            total = sum(apply_translations(f, lookup)[2] for f in html_files)
            if not missing:
                break
            print(f"[translate] ronda {round_no}: {total} unidades, "
                  f"{len(missing)} textos pendientes.", flush=True)
            batch_size = translate_pending(tr, cache, list(missing), batch_size, failed)
    except KeyboardInterrupt:
        print("\n⏸  Interrumpido. El progreso está guardado en Drive; "
              "vuelve a ejecutar esta celda para seguir.")
        return
    finally:
        cache.close()
    if failed:
        print(f"\n⚠ {len(failed)} textos quedan en el idioma original en el EPUB; "
              "vuelve a ejecutar esta celda para reintentarlos.")
    print(f"\n✅ Traducción terminada en {(time.time() - start) / 60:.1f} min. "
          "Ejecuta la celda 7 para armar el EPUB.")


# ─── Armado del EPUB ───────────────────────────────────────────────────────────

DOCTYPE_XHTML = ('<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.1//EN" '
                 '"http://www.w3.org/TR/xhtml11/DTD/xhtml11.dtd">\n')


PARTIAL_SUFFIX = " (parcial)"
# Agregado a cada CSS del libro: muchos EPUB no limitan el alto de las
# imágenes y, en una ventana baja, una imagen se corta y sigue en la página
# siguiente. En Apple Books 100vh es la ventana entera: se descuentan ~160px
# de su título y número de página, y 2em de márgenes del propio libro.
FIT_IMAGES_CSS = ("\n/* epub translator: las imágenes caben en la página */\n"
                  "img { max-width: 100%; max-height: calc(100vh - 160px - 2em); object-fit: contain; }\n")

# html.parser pasa a minúsculas nombres de etiquetas y atributos, pero en SVG
# (XML) importan las mayúsculas: sin viewBox/preserveAspectRatio la portada se
# deforma. Nombres SVG con mayúsculas, para restaurarlos al re-serializar.
SVG_NAMES = {n.lower(): n for n in (
    "viewBox preserveAspectRatio gradientUnits gradientTransform patternUnits "
    "patternContentUnits patternTransform clipPathUnits maskUnits maskContentUnits "
    "markerUnits markerWidth markerHeight refX refY textLength lengthAdjust "
    "startOffset spreadMethod stdDeviation baseFrequency numOctaves "
    "primitiveUnits filterUnits pathLength attributeName repeatCount "
    "linearGradient radialGradient clipPath foreignObject textPath "
    "feGaussianBlur feColorMatrix feOffset feBlend feFlood feComposite feMerge "
    "feMergeNode feImage feTurbulence animateTransform"
).split()}


def get_doctype(text: str) -> str:
    """DOCTYPE del archivo original (XHTML 1.1, EPUB3 `<!DOCTYPE html>`...),
    para no imponer uno distinto al del libro."""
    m = re.search(r"<!DOCTYPE[^>]*>", text, re.IGNORECASE)
    return (re.sub(r"\s+", " ", m.group(0)) + "\n") if m else DOCTYPE_XHTML


def fit_images(build_dir: Path):
    """Agrega FIT_IMAGES_CSS a todas las hojas de estilo del libro."""
    for css in build_dir.rglob("*.css"):
        with css.open("a", encoding="utf-8") as f:
            f.write(FIT_IMAGES_CSS)


def restore_svg_case(tree):
    """Devuelve a SVG las mayúsculas que html.parser le quitó (ver SVG_NAMES)."""
    for el in tree.iter("{http://www.w3.org/2000/svg}*"):
        ns, _, local = el.tag.rpartition("}")
        el.tag = ns + "}" + SVG_NAMES.get(local, local)
        for name in [a for a in el.attrib if a in SVG_NAMES]:
            el.set(SVG_NAMES[name], el.attrib.pop(name))


def fix_xhtml(text: str, doctype: str) -> bytes:
    """Re-serializa como XML válido (DOCTYPE, namespaces, self-closing).

    BeautifulSoup con html.parser es tolerante para leer, pero escribe HTML,
    no XHTML; los lectores EPUB (Apple Books en particular) exigen XML bien
    formado. lxml lo re-serializa como XML estricto."""
    # html.parser deja el DOCTYPE roto como texto "html PUBLIC ..."
    text = re.sub(r'^\s*(<!DOCTYPE[^>]*>|html\s+(?:PUBLIC|SYSTEM)\s[^\n]*)\n?', '',
                  re.sub(r'^\s*<\?xml[^>]*\?>\s*', '', text), flags=re.IGNORECASE)
    text = '<?xml version="1.0" encoding="utf-8"?>\n' + text
    try:
        tree = etree.fromstring(text.encode("utf-8"))
        restore_svg_case(tree)
        result = etree.tostring(tree, xml_declaration=True, encoding="utf-8")
    except etree.XMLSyntaxError as e:
        print(f"    ⚠ XML inválido, se guarda sin re-serializar: {e}", flush=True)
        result = text.encode("utf-8")
    end_decl = result.find(b"?>") + 2
    return result[:end_decl] + b"\n" + doctype.encode() + result[end_decl:]


def translate_text(text: str, cache: TranslationCache) -> str:
    """Traducción de un texto suelto (título, entrada del índice) o el original
    si no está en la caché."""
    lead, core, trail = split_ws(text)
    result = cache.get(core)
    return lead + result + trail if result else text


def update_opf(opf: Path, dropped: set[Path], cache: TranslationCache, tgt_lang: str,
               partial: bool) -> set[Path]:
    """Ajusta idioma, título, spine, manifest y guide. Devuelve los archivos
    del manifest marcados como nav (EPUB3)."""
    tree = etree.parse(str(opf))
    nav_files: set[Path] = set()
    dropped_ids = set()
    for item in list(tree.iterfind(".//{*}manifest/{*}item")):
        path = resolve_href(opf, item.get("href", ""))
        if path in dropped:
            dropped_ids.add(item.get("id"))
            item.getparent().remove(item)
        elif "nav" in (item.get("properties") or "").split():
            nav_files.add(path)
    for ref in list(tree.iterfind(".//{*}spine/{*}itemref")):
        if ref.get("idref") in dropped_ids:
            ref.getparent().remove(ref)
    for ref in list(tree.iterfind(".//{*}guide/{*}reference")):
        if resolve_href(opf, ref.get("href", "")) in dropped:
            ref.getparent().remove(ref)
    for lang in tree.iterfind(".//{*}metadata/{*}language"):
        lang.text = tgt_lang
    for title in tree.iterfind(".//{*}metadata/{*}title"):
        if title.text:
            title.text = translate_text(title.text, cache) + (PARTIAL_SUFFIX if partial else "")
    tree.write(str(opf), xml_declaration=True, encoding="utf-8")
    return nav_files


def update_ncx(ncx: Path, dropped: set[Path], cache: TranslationCache, partial: bool):
    """Traduce títulos del índice y quita entradas a capítulos descartados
    (sus sub-entradas que sigan existiendo suben un nivel)."""
    tree = etree.parse(str(ncx))
    for t in ncx_label_nodes(tree):
        t.text = translate_text(t.text, cache)
    if partial:
        for t in tree.iterfind(".//{*}docTitle/{*}text"):
            t.text = (t.text or "") + PARTIAL_SUFFIX
    # De adentro hacia afuera, para que los hijos ya estén resueltos. Un capítulo
    # descartado puede contener sub-entradas que sí se conservan (p. ej. la
    # portadilla agrupa "[Buch]" y "[Autor]"): suben un nivel en vez de perderse.
    for nav_point in reversed(list(tree.iterfind(".//{*}navPoint"))):
        content = nav_point.find("{*}content")
        if content is None or resolve_href(ncx, content.get("src", "")) not in dropped:
            continue
        parent = nav_point.getparent()
        pos = parent.index(nav_point)
        for child in reversed(nav_point.findall("{*}navPoint")):
            parent.insert(pos + 1, child)
        parent.remove(nav_point)
    for i, nav_point in enumerate(tree.iterfind(".//{*}navPoint"), 1):
        nav_point.set("playOrder", str(i))
    tree.write(str(ncx), xml_declaration=True, encoding="utf-8")


def prune_nav(soup, nav_file: Path, dropped: set[Path]):
    """Quita del nav EPUB3 los <li> que apuntan a capítulos descartados."""
    for a in soup.find_all("a", href=True):
        if resolve_href(nav_file, a["href"]) in dropped:
            li = a.find_parent("li")
            (li or a).decompose()


def assemble_epub(build_dir: Path, output_path: Path):
    """Comprime build/ como EPUB.

    La especificación exige que `mimetype` sea el primer archivo del zip y sin
    comprimir: así los lectores identifican el formato leyendo los primeros
    bytes. Se escribe a un .tmp y se renombra al final, para que un corte a
    mitad de camino nunca deje un EPUB roto en el lugar del bueno."""
    tmp = output_path.with_suffix(".epub.tmp")
    all_files = sorted(f for f in build_dir.rglob("*") if f.is_file() and f.name != "mimetype")
    with zipfile.ZipFile(tmp, "w") as zf:
        zf.writestr(zipfile.ZipInfo("mimetype"), b"application/epub+zip",
                    compress_type=zipfile.ZIP_STORED)
        for f in all_files:
            zf.write(f, str(f.relative_to(build_dir)), compress_type=zipfile.ZIP_DEFLATED)
    tmp.replace(output_path)
    size = output_path.stat().st_size / 1024
    print(f"[epub] {output_path.name} ({size:.0f} KB, {len(all_files) + 1} archivos)")


def build(input_epub: Path, output_epub: Path | None, src_lang: str, tgt_lang: str):
    work_dir, src_dir = book_dirs(input_epub)
    cpath = cache_path(work_dir, src_lang, tgt_lang)
    if not (src_dir / "mimetype").exists() or not cpath.exists():
        print(f"❌ No hay traducciones para {input_epub.name}. Ejecuta primero la celda 6.")
        return None
    # Las mismas notas que se usaron al traducir: forman parte de la clave de caché
    cache = TranslationCache(cpath, src_lang, tgt_lang, read_only=True,
                             notes=load_notes(input_epub, src_lang, tgt_lang))

    # Aplicar traducciones en orden de lectura y buscar el primer capítulo incompleto.
    # Un archivo sin texto (n == 0) cuenta como completo: no corta en la portada.
    # Textos que el modelo no logró traducir: están en la caché con el original
    # (el libro se arma igual), pero se avisa para que no pasen inadvertidos.
    untranslated: set[str] = set()

    def lookup(text: str) -> str | None:
        if cache.is_failed(text):
            untranslated.add(text)
        return cache.get(text)

    spine = spine_files(src_dir, include_rest=False)
    rendered = {}
    cut = len(spine)
    units_total = units_done = 0
    for i, f in enumerate(spine):
        soup, text, n, incomplete = apply_translations(f, lookup)
        rendered[f] = (soup, text, n)
        units_total += n
        units_done += n - incomplete
        if incomplete and cut == len(spine):
            cut = i
    for f in spine_files(src_dir):
        if f not in rendered:  # HTML fuera del spine (nav, etc.)
            soup, text, n, _ = apply_translations(f, lookup)
            rendered[f] = (soup, text, n)

    partial = cut < len(spine)
    kept = spine[:cut]
    if not any(rendered[f][2] for f in kept):
        print("❌ Todavía no hay ningún capítulo completamente traducido.")
        return None
    dropped = set(spine[cut:])

    if output_epub is None:
        suffix = f"{tgt_lang.upper()} parcial" if partial else tgt_lang.upper()
        output_epub = input_epub.parent / f"{input_epub.stem} ({suffix}).epub"

    build_dir = work_dir / "build"
    if build_dir.exists():
        shutil.rmtree(build_dir)
    shutil.copytree(src_dir, build_dir)
    fit_images(build_dir)

    def in_build(path: Path) -> Path:
        return build_dir / path.relative_to(src_dir)

    dropped_build = {in_build(p) for p in dropped}
    nav_files = update_opf(in_build(find_opf(src_dir)), dropped_build, cache, tgt_lang, partial)
    ncx = find_ncx(src_dir)
    if ncx and ncx.exists():
        update_ncx(in_build(ncx), dropped_build, cache, partial)

    for f, (soup, text, n) in rendered.items():
        target = in_build(f)
        if f in dropped:
            target.unlink()
            continue
        if target in nav_files:
            prune_nav(soup, target, dropped_build)
        elif not n:
            continue  # sin texto (p. ej. portada SVG): se deja el original intacto
        target.write_bytes(fix_xhtml(str(soup), get_doctype(text)))

    print(f"[build] {'Parcial' if partial else 'Completo'}: {len(kept)}/{len(spine)} "
          f"archivos del spine"
          + (f", corta antes de {spine[cut].relative_to(src_dir)}" if partial else "")
          + f". Unidades traducidas: {units_done}/{units_total} "
            f"({units_done / max(units_total, 1):.0%} del libro).")
    if untranslated:
        print(f"[build] ⚠ {len(untranslated)} textos quedaron en el idioma original porque "
              "el modelo no logró traducirlos; la celda 6 los reintenta al volver a ejecutarla.")
    assemble_epub(build_dir, output_epub)
    print(f"   Resultado: {output_epub}")
    return output_epub


print("Funciones listas.")


## 4. Token de Hugging Face
Lee `HF_TOKEN` de los *Secretos* de Colab (🔑). Si no está, pide iniciar sesión.


In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Sesión de Hugging Face iniciada con el secreto HF_TOKEN.")
except Exception:
    HF_TOKEN = None
    print("No encontré el secreto HF_TOKEN; inicia sesión manualmente:")
    login()


## 5. Cargar el modelo y probarlo

Carga el modelo de la celda 1. Si no entra completo en la GPU, se carga cuantizado a 8 o 4 bits (bitsandbytes): en una L4 (24 GB), TranslateGemma 12B va en 8 bits y Gemma 4 26B-A4B en 4 bits. Gemma 4 es un modelo general: recibe instrucciones de traductor literario en vez de la plantilla de TranslateGemma.

Usa **bf16** si la GPU lo soporta (L4, A100) y **fp16** si no (T4). Gemma fue entrenado en bf16; en fp16 a veces sus activaciones se desbordan y la salida sale vacía o sin sentido. La prueba de abajo lo detecta: si falla en una T4, cambia a una GPU L4/A100.

**Por qué es rápido**
- **Lotes:** generar texto está limitado por la memoria (cada token exige leer todos los pesos), así que traducir varios párrafos a la vez aprovecha cada lectura de los pesos para producir un token de cada párrafo.
- **Token de parada `<end_of_turn>`:** el modelo cierra su respuesta con ese token, que no está declarado como fin. Sin agregarlo, seguiría generando texto inútil hasta el límite.
- **Límite de tokens según el largo** del párrafo (×2.5 + 64): nunca corta una traducción normal y acota lo que cuesta un bucle.


In [ ]:
import re

import torch
from transformers import (AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor,
                          BitsAndBytesConfig)


def pick_quantization(model_id: str) -> str:
    """"none" o "4bit" según la memoria de la GPU ("8bit" solo si se elige en
    QUANTIZATION: bitsandbytes genera en 8 bits varias veces más lento que en
    4 bits, así que no conviene como opción automática).
    Estimación: ~2.1 GB por cada mil millones de parámetros en bf16, ~0.6 en
    4 bits, más ~4 GB para los párrafos en curso."""
    if QUANTIZATION != "auto":
        return QUANTIZATION
    m = re.search(r"(\d+)[bB]", model_id.split("/")[-1])
    params = float(m.group(1)) if m else 4.0
    gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    return "none" if params * 2.1 + 4 <= gpu_gb else "4bit"


class HFTranslator:
    """TranslateGemma o un modelo general (Gemma 4) con transformers, por lotes."""

    def __init__(self, model_id: str, src_lang: str, tgt_lang: str, token=None):
        if not torch.cuda.is_available():
            raise RuntimeError("No hay GPU. Entorno de ejecución → Cambiar tipo → T4 GPU.")
        # bf16 nativo desde compute capability 8.0 (Ampere: A100, L4). La T4 es 7.5.
        # (torch.cuda.is_bf16_supported() devuelve True en T4 por emulación, que es lenta.)
        self.dtype = torch.bfloat16 if torch.cuda.get_device_capability() >= (8, 0) else torch.float16
        self.src_lang, self.tgt_lang = src_lang, tgt_lang
        self.quantization = pick_quantization(model_id)
        quant_config = None
        if self.quantization == "8bit":
            quant_config = BitsAndBytesConfig(load_in_8bit=True)
        elif self.quantization == "4bit":
            quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                              bnb_4bit_compute_dtype=self.dtype,
                                              bnb_4bit_use_double_quant=True)
        self.processor = AutoProcessor.from_pretrained(model_id, token=token)
        self.tok = getattr(self.processor, "tokenizer", self.processor)
        self.tok.padding_side = "left"  # generación por lotes: el texto nuevo va a la derecha
        kwargs = dict(dtype=self.dtype, device_map="cuda", token=token,
                      quantization_config=quant_config)
        try:  # Gemma 3/4 son multimodales; si el modelo no lo es, se carga como texto
            self.model = AutoModelForImageTextToText.from_pretrained(model_id, **kwargs)
        except ValueError:
            self.model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        self.model.eval()
        # Fin de turno: <end_of_turn> (Gemma 3) o <turn|> (Gemma 4), además de los
        # eos del tokenizer y de la configuración de generación del modelo
        stop = {self.tok.eos_token_id, self.tok.convert_tokens_to_ids(END_OF_TURN),
                self.tok.convert_tokens_to_ids("<turn|>")}
        gen_eos = getattr(self.model.generation_config, "eos_token_id", None)
        stop.update(gen_eos if isinstance(gen_eos, list) else [gen_eos])
        unk = self.tok.unk_token_id
        self.stop_ids = sorted(t for t in stop if isinstance(t, int) and t != unk)
        self._n_tokens: dict[str, int] = {}

    def prompt(self, text: str, notes: str = "") -> str:
        if MODEL_KIND == "chat":
            system = CHAT_SYSTEM_PROMPT.format(src=lang_name(self.src_lang),
                                               tgt=lang_name(self.tgt_lang))
            if notes:
                system += "\n\n" + notes
            messages = [{"role": "system", "content": [{"type": "text", "text": system}]},
                        {"role": "user", "content": [{"type": "text", "text": text}]}]
            # enable_thinking=False: sin esto Gemma 4 "piensa" antes de responder
            # y gasta el límite de tokens
            return self.processor.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True,
                                                      enable_thinking=False)
        messages = [{"role": "user", "content": [{
            "type": "text", "source_lang_code": self.src_lang,
            "target_lang_code": self.tgt_lang, "text": text}]}]
        rendered = self.processor.apply_chat_template(messages, tokenize=False,
                                                      add_generation_prompt=True)
        return insert_notes(rendered, notes) if notes else rendered

    def n_tokens(self, text: str) -> int:
        if text not in self._n_tokens:
            self._n_tokens[text] = len(self.tok(text, add_special_tokens=False).input_ids)
        return self._n_tokens[text]

    def translate_batch(self, texts: list[str], notes: list[str] | None = None,
                        retry: bool = False) -> list[tuple[str, bool]]:
        """Devuelve [(traducción, terminó_bien)] en el mismo orden que `texts`.
        `notes[i]` es el bloque de notas de traducción de texts[i] ("" si no hay)."""
        notes = notes or [""] * len(texts)
        # El prompt ya incluye <bos>: no agregar tokens especiales otra vez
        enc = self.tok([self.prompt(t, n) for t, n in zip(texts, notes)], return_tensors="pt",
                       padding=True, add_special_tokens=False).to(self.model.device)
        # El reintento tiene el doble de margen: una traducción legítimamente
        # larga que se cortó no debe cortarse otra vez con el mismo límite.
        limits = [(int(self.n_tokens(t) * MAX_TOKENS_FACTOR) + MAX_TOKENS_MARGIN) * (1 + retry)
                  for t in texts]
        with torch.inference_mode():
            out = self.model.generate(
                **enc, max_new_tokens=max(limits), do_sample=False,
                eos_token_id=self.stop_ids, pad_token_id=self.tok.pad_token_id,
                repetition_penalty=RETRY_REPETITION_PENALTY if retry else None)
        results = []
        for row, limit in zip(out[:, enc["input_ids"].shape[1]:].tolist(), limits):
            stop_at = next((i for i, t in enumerate(row) if t in self.stop_ids), None)
            # Todo el lote usa el max_tokens del más largo: un párrafo que lo superó
            # sin terminar se trata como cortado aunque el lote siguiera.
            finished = stop_at is not None and stop_at <= limit
            text = self.tok.decode(row[:stop_at], skip_special_tokens=False)
            results.append((clean_output(re.sub(r"<pad>|<eos>|<bos>", "", text)), finished))
        return results

    def free_memory(self):
        gc.collect()
        torch.cuda.empty_cache()


t0 = time.time()
tr = HFTranslator(MODEL_ID, SRC_LANG, TGT_LANG, HF_TOKEN)
gpu = torch.cuda.get_device_name()
print(f"{MODEL_ID} cargado en {gpu} ({str(tr.dtype).split('.')[-1]}, "
      f"cuantización: {tr.quantization}) en {time.time() - t0:.0f}s")

# Prueba rápida
samples = {
    "de": ["Die Sonne schien, und ein leichter Wind wehte über den Fluss.",
           "Sie kannte einen Filmtitel, *Große Freiheit Nummer 7*. Ein schöner Titel."],
    "en": ["The sun was shining, and a light wind blew over the river.",
           "She knew a film called *The Great Escape*. A beautiful title."],
}.get(SRC_LANG.split("-")[0])
if samples:
    t0 = time.time()
    outputs = tr.translate_batch(samples)
    for s, (o, ok) in zip(samples, outputs):
        print(f"\n{s}\n→ {o}")
    bad = [o for s, (o, ok) in zip(samples, outputs) if not ok or output_problem(s, o)]
    print(f"\n({time.time() - t0:.1f}s)")
    if bad:
        print("⚠ La prueba dio salidas vacías o sin sentido. Si estás en fp16 (T4), "
              "cambia a una GPU L4 o A100 (Colab Pro) y vuelve a ejecutar desde la celda 2.")
    else:
        print("✓ El modelo traduce correctamente.")
else:
    print("(Sin frases de prueba para este idioma origen; se omite la prueba.)")


## 6. Traducir
Traduce en orden de lectura y guarda cada lote en Drive. Puedes detenerla con ⏹ en cualquier momento y volver a ejecutarla: solo traduce lo que falta.


In [ ]:
translate_book(tr, EPUB, SRC_LANG, TGT_LANG, BATCH_SIZE)


## 7. Armar el EPUB (completo o parcial)
Puedes ejecutarla en cualquier momento, incluso con la traducción a medias: arma un EPUB con los capítulos ya traducidos, cortando en el primero incompleto. El archivo queda en Drive (en la carpeta del EPUB original) y además se descarga.


In [ ]:
result = build(EPUB, None, SRC_LANG, TGT_LANG)
if result:
    files.download(str(result))
